# 🏎️ Team ASTRA — Llama 3.1 8B Fine-Tuning Pipeline
This notebook contains the complete, ready-to-run code to load your Firebase training dataset, perform LoRA fine-tuning using Unsloth, export the model to GGUF format, load it into Ollama, and expose the server to the internet using localtunnel.

### ⚡ Step 1: Install GPU-Accelerated Unsloth & PEFT Dependencies

In [ ]:
# Install Unsloth and compatible libraries
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

### 🔑 Step 2: Configure Firebase Credentials
Configure your Firebase parameters below to load the dataset generated from the ASTRA web app.

In [ ]:
import os
import requests
import json
import torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

# Set Firebase credentials
FIREBASE_DATABASE_URL = "https://studio-1045950084-89865-default-rtdb.asia-southeast1.firebasedatabase.app"
FIREBASE_DATABASE_SECRET = "nbN32sF35ZGFoP3IdVaGkVb5t9gW5NFj3V7Gu7rY"

# Optional: To save directly to Hugging Face Hub
HUGGING_FACE_TOKEN = ""
HF_OUTPUT_REPOSITORY = "your_username/astra-llama-3.1-8b"

### 📡 Step 3: Fetch Dataset from Firebase RTDB

In [ ]:
def download_dataset():
    print("📡 Downloading training dataset from Firebase Realtime Database...")
    url = f"{FIREBASE_DATABASE_URL.rstrip('/')}/finetuning/dataset.json?auth={FIREBASE_DATABASE_SECRET}"
    response = requests.get(url)
    
    if response.status_code != 200:
        raise Exception(f"❌ Failed to fetch dataset: HTTP {response.status_code}\n{response.text}")
        
    data = response.json()
    print(f"✅ Downloaded {len(data)} instruction-response pairs successfully!")
    return data

dataset_json = download_dataset()

### 🚀 Step 4: Load Base Model and Apply LoRA Adapters

In [ ]:
max_seq_length = 2048 
dtype = None 
load_in_4bit = True 

print("🚀 Loading base model (Llama-3.1-8B-Instruct-bnb-4bit)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/meta-llama-3.1-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Apply PEFT / LoRA target configuration
print("🔧 Configuring LoRA parameters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 📝 Step 5: Format Prompts & Run Fine-Tuning

In [ ]:
# Format dataset into Llama 3.1 conversational template
ASTRA_SYSTEM_PROMPT = (
    "You are ASTRA AI, the expert intelligence assistant for Team ASTRA "
    "(Sri Krishna College of Engineering and Technology's Solar Electric Vehicle Team). "
    "Use your extensive engineering knowledge about steering, suspension, brakes, "
    "transmission, autonomous systems, electrical schemas, budgets, and rulebooks to answer "
    "queries accurately."
)

def format_prompts(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = (
            "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
            f"{ASTRA_SYSTEM_PROMPT}<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n\n"
            f"{instruction}<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n\n"
            f"{output}<|eot_id|>"
        )
        texts.append(text)
    return { "text" : texts }

raw_dataset = Dataset.from_list(dataset_json)
dataset = raw_dataset.map(format_prompts, batched = True)

print("📝 Starting training configuration...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, 
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🏋️‍‍♂️ Training model...")
trainer.train()
print("🎉 Training completed!")

### 💾 Step 6: Save GGUF and Host inside Colab

In [ ]:
# Quantize and Export to GGUF format
print("💾 Saving fine-tuned model to GGUF (Q4_K_M)...")
model.save_pretrained_gguf("astra_model", tokenizer, quantization_method = "q4_k_m")

### 🚀 Step 7: Install Ollama & Import Model

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
import subprocess
import time
print("🚀 Starting Ollama background server...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5) # Wait for startup

# Create a Modelfile pointing to local GGUF
with open("Modelfile", "w") as f:
    f.write("FROM ./astra_model-unsloth.Q4_K_M.gguf")

# Import model into Ollama
!ollama create astra-model -f Modelfile
print("✅ astra-model registered inside Ollama successfully!")

### 📡 Step 8: Expose Model to Internet via Localtunnel
Run the cell below to generate your unique localtunnel URL. Copy the printed URL, add `/v1/chat/completions` at the end, and paste it into the ASTRA Chatbot settings panel!

In [ ]:
# Install localtunnel globally
!npm install -g localtunnel

# Expose Ollama (port 11434) to the internet
print("📡 Exposing Ollama on port 11434...")
!lt --port 11434